In [1]:
import os
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import statsmodels.api as sm

import numpy as np
import pandas as pd
from scipy.interpolate import griddata
from sklearn.linear_model import LinearRegression

import plotly.graph_objects as go
from plotly.subplots import make_subplots

from scipy.interpolate import CloughTocher2DInterpolator
from scipy.interpolate import griddata

from math import ceil


In [8]:
#configuration
@dataclass
class PipelineConfig:
    subject_ids: Tuple[int, ...] = (1, 2, 4, 5, 8, 9, 10, 11, 12, 14)
    #subject_ids: Tuple[int, ...] = (1,)

    out_dir: str = "individuals"

    #physio_signals: Tuple[str, ...] = ("glucose", "acc_x", "acc_y", "acc_z", "eda", "temp", "bvp")
    #physio_signals: Tuple[str, ...] = ("glucose", "eda", "temp", "bvp")
    #nutrients: Tuple[str, ...] = ("sugar_protein_ratio", "sugar_carb_ratio", "sugar")

    physio_signals: Tuple[str, ...] = ("eda", "temp", "bvp")
    nutrients: Tuple[str, ...] = ("sugar", )
    
    feature_dir: str = "./25min_backshift/"
    file_pattern: str = "segmented_df_{subject_id}_7200.pkl"
    #file_pattern: str = "segmented_df_1_7200.pkl"
    post_meal_epochs: int = 24      # number of postprandial epochs
    epoch_window_mins: int = 5      # epoch size (minutes)
    outlier_threshold: float = 75.0 # absolute threshold on delta AUC (can be signal-specific if needed)

    # Plotting output directories
    #out_dir_2d: str = out_dir + "/2d_scatter"
    #out_dir_3d: str = out_dir + "/3d_scatter"
    out_dir_surface: str = out_dir + "/3d_surface"

In [9]:
#Subject-level Processor Class
class PostprandialProcessor:
    """
    Handles per-subject, per-signal processing:
    - aligns segments to meal times
    - computes baseline and epoch-level AUC
    - computes ΔAUC and Δvalue
    - extracts nutrient fields and time-of-day labels
    """

    def __init__(self, config: PipelineConfig):
        self.config = config

    @staticmethod
    def classify_time_of_day(meal_time: pd.Timestamp) -> str:
        hour = meal_time.hour
        if 6 <= hour < 12:
            return "morning"
        elif 12 <= hour < 17:  # Why set it to 20?
            return "afternoon"
        else:
            return "night"

    def _compute_auc(
        self,
        window: pd.DataFrame,
        value_col: str,
        start_time: pd.Timestamp
    ) -> float:
        """Compute AUC over a window using trapezoidal rule."""
        times = (window.index - start_time).total_seconds() / 60  # minutes
        values = window[value_col].values
        #return float(np.trapezoid(values, times))
        return float(np.trapz(values, times))

    def process_subject(
        self,
        subject_id: int,
        physio_signal: str
    ) -> pd.DataFrame:
        """
        Process one subject and one physiological signal.
        Returns a DataFrame with per-epoch postprandial features for all meals.
        """
        file_path = os.path.join(
            self.config.feature_dir,
            self.config.file_pattern.format(subject_id=subject_id)
        )

        df = pd.read_pickle(file_path)
        segment_col = f"{physio_signal}_segments"
        value_name = physio_signal

        # Pre-sort meal_times for next-meal lookup
        df = df.sort_values("timestamp")
        meal_times_sorted = pd.to_datetime(df["timestamp"]).tolist()

        results: List[Dict] = []

        for _, row in df.iterrows():
            try:
                meal_time = pd.to_datetime(row["timestamp"])
                time_of_day = self.classify_time_of_day(meal_time)

                # Next meal time (if any)
                next_meal_time: Optional[pd.Timestamp] = None
                future_meals = [t for t in meal_times_sorted if t > meal_time]
                if future_meals:
                    next_meal_time = future_meals[0]

                seg_df = row[segment_col]

                # --- Segment validity checks ---
                if not isinstance(seg_df, pd.DataFrame) or seg_df.empty:
                    continue
                if value_name not in seg_df.columns:
                    continue
                if seg_df[value_name].isna().mean() > 0.5:
                    continue

                if not isinstance(seg_df.index, pd.DatetimeIndex):
                    seg_df.index = pd.to_datetime(seg_df.index)

                # Resample to 5-min intervals
                seg_df = seg_df.resample("5T").mean()

                # Find baseline index closest to meal_time
                time_diffs = np.abs((seg_df.index - meal_time).total_seconds())
                if len(time_diffs) == 0:
                    continue
                meal_idx = int(np.argmin(time_diffs))

                # Baseline epoch: [baseline_start, baseline_start + epoch_window]
                baseline_start = seg_df.index[meal_idx]
                baseline_end = baseline_start + pd.Timedelta(minutes=self.config.epoch_window_mins)

                baseline_window = seg_df.loc[
                    (seg_df.index >= baseline_start) & (seg_df.index <= baseline_end)
                ]

                if len(baseline_window) < 2:
                    continue

                baseline_auc = self._compute_auc(baseline_window, value_name, baseline_start)
                baseline_value = baseline_window[value_name].iloc[0]

                # Precompute nutrient fields
                total_carb = row.get("total_carb", np.nan)
                sugar = row.get("sugar", np.nan)
                protein = row.get("protein", np.nan)
                calories = row.get("calorie", np.nan)
                total_fat = row.get("total_fat", np.nan)

                carb_fat_ratio = (
                    total_carb / total_fat
                    if (pd.notna(total_fat) and total_fat > 0) else np.nan
                )
                sugar_carb_ratio = (
                    sugar / total_carb
                    #if (pd.notna(total_carb) and total_carb > 0) else np.nan
                    if (pd.notna(total_carb) and total_carb > 0 and (sugar / total_carb) <= 1) else np.nan
                )
                sugar_protein_ratio = (
                    sugar / protein
                    if (pd.notna(protein) and protein > 0) else np.nan
                )

                # Epoch loop: start from epoch 1 (post-baseline)
                previous_auc = baseline_auc
                actual_epochs_processed = 0

                for epoch_num in range(1, self.config.post_meal_epochs + 1):
                    epoch_start = baseline_start + pd.Timedelta(
                        minutes=epoch_num * self.config.epoch_window_mins
                    )
                    epoch_end = epoch_start + pd.Timedelta(minutes=self.config.epoch_window_mins)

                    # Stop if the epoch starts after the next meal
                    if next_meal_time is not None and epoch_start >= next_meal_time:
                        break

                    window = seg_df.loc[
                        (seg_df.index >= epoch_start) & (seg_df.index <= epoch_end)
                    ]
                    if len(window) < 2:
                        continue

                    current_auc = self._compute_auc(window, value_name, epoch_start)
                    delta_auc = current_auc - baseline_auc
                    delta_value = current_auc - previous_auc

                    # Simple outlier rule (can be replaced by z-score, per-signal thresholds, etc.)
                    if abs(delta_value) > self.config.outlier_threshold:
                        continue

                    results.append({
                        "subject_id": subject_id,
                        "meal_time": meal_time,
                        "time_of_day": time_of_day,
                        "signal": value_name,
                        "epoch_num": epoch_num,
                        "start_time": epoch_start,
                        "end_time": epoch_end,
                        "auc": current_auc,
                        "delta_auc": delta_auc,
                        "delta_value": delta_value,
                        "baseline_auc": baseline_auc,
                        "baseline_value": baseline_value,
                        "current_value": window[value_name].iloc[-1],
                        "total_carb": total_carb,
                        "sugar": sugar,
                        "protein": protein,
                        "calories": calories,
                        "carb_fat_ratio": carb_fat_ratio,
                        "sugar_carb_ratio": sugar_carb_ratio,
                        "sugar_protein_ratio": sugar_protein_ratio,
                        "actual_epochs": epoch_num
                    })

                    previous_auc = current_auc
                    actual_epochs_processed = epoch_num

                print(
                    f"Subject {subject_id}, signal {value_name}: "
                    f"processed {actual_epochs_processed} epochs for meal at {meal_time}"
                )

            except Exception as e:
                print(f"Error processing subject {subject_id}, meal at {row['timestamp']}: {str(e)}")
                continue

        if not results:
            return pd.DataFrame()

        results_df = pd.DataFrame(results)
        results_df["mins_since_meal"] = (
            results_df["start_time"] - results_df["meal_time"]
        ).dt.total_seconds() / 60

        print(f"\nProcessed {value_name} for subject {subject_id}:")
        print(f"  Total meals: {results_df['meal_time'].nunique()}")
        print("  Time-of-day distribution:")
        print(results_df["time_of_day"].value_counts())
        print("  Epochs per meal:")
        print(results_df.groupby("meal_time")["actual_epochs"].max().describe())

        return results_df


In [10]:
#Population-level Pipeline
class PostprandialPipeline:
    def __init__(self, config: PipelineConfig):
        self.config = config
        self.processor = PostprandialProcessor(config)

    def run_for_signal(self, physio_signal: str) -> Dict[str, pd.DataFrame]:
        """
        Run the pipeline for a single physiological signal across all subjects.
        Returns a dict with population-level DataFrames stratified by time-of-day.
        """
        population_data = pd.DataFrame()
        population_morning = pd.DataFrame()
        population_afternoon = pd.DataFrame()
        population_night = pd.DataFrame()

        for subject_id in self.config.subject_ids:
            subject_df = self.processor.process_subject(subject_id, physio_signal)
            if subject_df.empty:
                continue

            population_data = pd.concat([population_data, subject_df], ignore_index=True)

            population_morning = pd.concat(
                [population_morning, subject_df[subject_df["time_of_day"] == "morning"]],
                ignore_index=True
            )
            population_afternoon = pd.concat(
                [population_afternoon, subject_df[subject_df["time_of_day"] == "afternoon"]],
                ignore_index=True
            )
            population_night = pd.concat(
                [population_night, subject_df[subject_df["time_of_day"] == "night"]],
                ignore_index=True
            )
            
        """
            
        fileAll = self.config.out_dir + "/" + physio_signal + "_all.csv"
        fileMorning = self.config.out_dir + "/" + physio_signal + "_morning.csv"
        fileAfternoon = self.config.out_dir + "/" + physio_signal + "_afternoon.csv"
        fileNight = self.config.out_dir + "/" + physio_signal + "_night.csv"
        
        population_data.to_csv(fileAll)
        population_morning.to_csv(fileMorning)
        population_afternoon.to_csv(fileAfternoon)
        population_night.to_csv(fileNight)
        
        """

        return {
            "All Data": population_data,
            "Morning": population_morning,
            "Afternoon": population_afternoon,
            "Night": population_night
        }


In [11]:
#Visualization Class
class PostprandialVisualizer:
    def __init__(self, config: PipelineConfig):
        self.config = config
        #os.makedirs(self.config.out_dir_2d, exist_ok=True)
        #os.makedirs(self.config.out_dir_3d, exist_ok=True)
        os.makedirs(self.config.out_dir_surface, exist_ok=True)

    def plot_2d_by_time_of_day(
        self,
        dataframes: Dict[str, pd.DataFrame],
        physio_signal: str,
        nutrient: str
    ):
        """Vertical stack of 2D nutrient vs ΔAUC plots (Morning / Afternoon / Night / All)."""
        df_all = dataframes["All Data"]
        if df_all.empty:
            print(f"No data for 2D plotting: {physio_signal}, {nutrient}")
            return

        # Determine shared color scale
        all_auc = np.concatenate([
            df["delta_auc"].values
            for df in dataframes.values() if not df.empty
        ])
        cmin, cmax = np.nanmin(all_auc), np.nanmax(all_auc)

        fig = make_subplots(
            rows=4, cols=1,
            subplot_titles=[
                f"Morning: {nutrient} vs ΔAUC",
                f"Afternoon: {nutrient} vs ΔAUC",
                f"Night: {nutrient} vs ΔAUC",
                f"All Data: {nutrient} vs ΔAUC"
            ],
            shared_xaxes=True,
            vertical_spacing=0.1
        )

        order = ["Morning", "Afternoon", "Night", "All Data"]
        for i, label in enumerate(order, start=1):
            df = dataframes[label]
            if df.empty:
                continue
            clean_df = df.dropna(subset=[nutrient, "delta_auc"])
            x_vals = clean_df[nutrient]
            y_vals = clean_df["delta_auc"]

            fig.add_trace(
                go.Scatter(
                    x=x_vals,
                    y=y_vals,
                    mode="markers",
                    marker=dict(
                        size=8,
                        opacity=0.7,
                        color=y_vals,
                        colorscale="Inferno",
                        cmin=cmin,
                        cmax=cmax,
                        showscale=(i == 4)
                    ),
                    name=label,
                    showlegend=False
                ),
                row=i, col=1
            )

            # Linear trendline
            if len(x_vals) > 1:
                model = LinearRegression().fit(x_vals.values.reshape(-1, 1), y_vals)
                trend_x = np.linspace(x_vals.min(), x_vals.max(), 100)
                trend_y = model.predict(trend_x.reshape(-1, 1))

                fig.add_trace(
                    go.Scatter(
                        x=trend_x,
                        y=trend_y,
                        mode="lines",
                        line=dict(color="red", width=2),
                        name=f"Slope: {model.coef_[0]:.4f}",
                        showlegend=False
                    ),
                    row=i, col=1
                )

        fig.update_layout(
            title_text=f"{nutrient} vs ΔAUC by Time of Day ({physio_signal})",
            height=1200,
            template="plotly_dark"
        )

        for i in range(1, 5):
            fig.update_yaxes(title_text="ΔAUC", row=i, col=1)
        fig.update_xaxes(title_text=f"{nutrient}", row=4, col=1)

        out_path = os.path.join(
            self.config.out_dir_2d,
            f"{physio_signal}_{nutrient}_vertical_population.html"
        )
        fig.write_html(out_path)
        print(f"Saved 2D plot: {out_path}")

    def plot_3d_scatter(
        self,
        dataframes: Dict[str, pd.DataFrame],
        physio_signal: str,
        nutrient: str
    ):
        
        """3D scatter: nutrient vs epoch_num vs ΔAUC, per time-of-day."""
        
        for time_label, df in dataframes.items():
            if df.empty:
                continue

            clean_df = df.dropna(subset=[nutrient, "epoch_num", "delta_auc"])
            if clean_df.empty:
                continue

            fig = go.Figure()
            fig.add_trace(
                go.Scatter3d(
                    x=clean_df[nutrient],
                    y=clean_df["epoch_num"],
                    z=clean_df["delta_auc"],
                    mode="markers",
                    marker=dict(
                        size=5,
                        color=clean_df["delta_auc"],
                        colorscale="Inferno",
                        opacity=0.8
                    )
                )
            )

            fig.update_layout(
                title=f"{time_label}: {nutrient} vs Epoch vs ΔAUC ({physio_signal})",
                scene=dict(
                    xaxis_title=f"{nutrient}",
                    yaxis_title="Epoch Number",
                    zaxis_title="ΔAUC",
                    camera=dict(eye=dict(x=1.5, y=1.5, z=0.5))
                ),
                height=800
            )

            out_path = os.path.join(
                self.config.out_dir_3d,
                f"{physio_signal}_{nutrient}_3d_{time_label.replace(' ', '_')}_population.html"
            )
            fig.write_html(out_path)
            print(f"Saved 3D scatter: {out_path}")
            

    def plot_3d_surface(
        self,
        dataframes: Dict[str, pd.DataFrame],
        physio_signal: str,
        nutrient: str
    ):
        """3D surface: nutrient × epoch_num → ΔAUC, per time-of-day,
        with a shared color scale across all panels for this physio_signal + nutrient.
        """

        # ---- 1. Compute a global ΔAUC range across all time-of-day dataframes ----
        all_z_list = []
        for df in dataframes.values():
            if df is None or df.empty:
                continue
            if nutrient not in df.columns:
                continue
            clean_df = df.dropna(subset=[nutrient, "epoch_num", "delta_auc"])
            if clean_df.empty:
                continue
            all_z_list.append(clean_df["delta_auc"].values)

        if not all_z_list:
            print(f"No valid ΔAUC values for 3D surface: {physio_signal}, {nutrient}")
            return

        all_z = np.concatenate(all_z_list)

        # Option A: robust range (recommended, like your 5–95% per-plot)
        z_min, z_max = np.nanpercentile(all_z, [5, 95])

        # Option B (if you prefer to include full range, comment A and uncomment this):
        # z_min, z_max = np.nanmin(all_z), np.nanmax(all_z)

        # Safety: avoid degenerate range
        if z_min == z_max:
            # give it a tiny span to avoid Plotly complaints
            z_min -= 1.0
            z_max += 1.0

        # ---- 2. Create a surface per time-of-day using the shared z_min/z_max ----
        for time_label, df in dataframes.items():
            if df.empty or len(df) < 15:
                continue

            if nutrient not in df.columns:
                continue

            clean_df = df.dropna(subset=[nutrient, "epoch_num", "delta_auc"])
            if len(clean_df) < 10:
                continue

            x = clean_df[nutrient].values
            y = clean_df["epoch_num"].values
            z = clean_df["delta_auc"].values

            x_min, x_max = np.nanmin(x), np.nanmax(x)
            y_min, y_max = np.nanmin(y), np.nanmax(y)

            # Skip if no spread
            if x_min == x_max or y_min == y_max:
                print(f"Skipping surface for {time_label}, {physio_signal}, {nutrient}: "
                      f"insufficient spread in x or y.")
                continue

            xi = np.linspace(max(x_min, 0), x_max, 50)
            yi = np.linspace(max(y_min, 0), y_max, 50)
            xi, yi = np.meshgrid(xi, yi)
            
            interp_method = "cubic"  # or "cubic" or "nearest"
            #interp = CloughTocher2DInterpolator(np.column_stack((x, y)), z)
            
            try:
                zi = griddata((x, y), z, (xi, yi), method=interp_method)
                #zi = interp(xi, yi)

                if np.isnan(zi).any():
                    zi_nearest = griddata((x, y), z, (xi, yi), method="nearest")
                    zi = np.where(np.isnan(zi), zi_nearest, zi)

                if np.all(np.isnan(zi)):
                    print(f"Surface for {time_label}, {physio_signal}, {nutrient} is all NaNs after interpolation.")
                    continue

                # Use the GLOBAL z_min/z_max for clipping and color scale
                zi = np.clip(zi, z_min, z_max)

            except Exception as e:
                print(f"Could not create surface for {time_label}, {physio_signal}, {nutrient}: {e}")
                continue

            fig_surface = go.Figure(
                data=[go.Surface(
                    x=xi,
                    y=yi,
                    z=zi,
                    colorscale="Inferno",
                    opacity=0.9,
                    contours_z=dict(
                        show=True,
                        usecolormap=True,
                        project_z=True
                    ),
                    showscale=True,
                    cmin=z_min,
                    cmax=z_max
                )]
            )

            fig_surface.update_layout(
                title=(f"{time_label}: {nutrient} vs Epoch vs ΔAUC (Surface) "
                       f"({physio_signal})<br><sup>Interpolated from {len(clean_df)} points</sup>"),
                scene=dict(
                    xaxis_title=f"{nutrient}",
                    yaxis_title="Epoch Number",
                    zaxis_title="ΔAUC",
                    zaxis=dict(range=[z_min, z_max]),
                    camera=dict(eye=dict(x=1.5, y=1.5, z=0.5))
                ),
                height=800,
                margin=dict(l=50, r=50, b=50, t=100)
            )

            out_path = os.path.join(
                self.config.out_dir_surface,
                f"{physio_signal}_{nutrient}_surface_{time_label.replace(' ', '_')}_population.html"
            )
            fig_surface.write_html(out_path)
            print(f"Saved 3D surface: {out_path}")
            
    def plot_3d_surface_by_subject(
        self,
        df_all: pd.DataFrame,
        physio_signal: str,
        nutrient: str,
        time_label: str = "All Data",
        max_cols: int = 3    # how many subjects per row
    ):
        """
        3D surface: nutrient × epoch_num → ΔAUC, for each subject,
        arranged in a grid of 3D subplots with shared axis ranges and color scale.

        df_all must contain columns:
            - 'subject_id'
            - nutrient (e.g., 'sugar')
            - 'epoch_num'
            - 'delta_auc'
        """

        # -------- 0. Basic checks --------
        if df_all is None or df_all.empty:
            print(f"No data for per-subject 3D surfaces: {physio_signal}, {nutrient}")
            return

        required_cols = {"subject_id", nutrient, "epoch_num", "delta_auc"}
        missing = required_cols - set(df_all.columns)
        if missing:
            print(f"Missing columns {missing} in df_all for per-subject surfaces.")
            return

        # Drop NaNs and filter to valid rows
        df_clean = df_all.dropna(subset=["subject_id", nutrient, "epoch_num", "delta_auc"])
        if df_clean.empty:
            print(f"No valid rows after dropna for per-subject surfaces: {physio_signal}, {nutrient}")
            return

        # -------- 1. Group by subject and filter small groups --------
        subject_groups = {}
        for subj, df_sub in df_clean.groupby("subject_id"):
            if len(df_sub) >= 10:   # same idea as your population threshold
                subject_groups[subj] = df_sub

        if not subject_groups:
            print(f"No subjects with enough data for per-subject surfaces: {physio_signal}, {nutrient}")
            return

        # -------- 2. Compute global ranges across all subjects --------
        # Global z range (ΔAUC)
        all_z = np.concatenate([g["delta_auc"].values for g in subject_groups.values()])
        z_min, z_max = np.nanpercentile(all_z, [5, 95])   # robust range, like your code
        if z_min == z_max:
            z_min -= 1.0
            z_max += 1.0

        # Global x and y ranges
        all_x = np.concatenate([g[nutrient].values for g in subject_groups.values()])
        all_y = np.concatenate([g["epoch_num"].values for g in subject_groups.values()])
        x_min, x_max = np.nanmin(all_x), np.nanmax(all_x)
        y_min, y_max = np.nanmin(all_y), np.nanmax(all_y)

        # Safety
        if x_min == x_max or y_min == y_max:
            print("Insufficient global spread in x or y for per-subject surfaces.")
            return

        # Create a common interpolation grid (same for all subjects)
        xi = np.linspace(max(x_min, 0), x_max, 50)
        yi = np.linspace(max(y_min, 0), y_max, 50)
        xi_mesh, yi_mesh = np.meshgrid(xi, yi)

        # -------- 3. Prepare subplot grid --------
        n_subjects = len(subject_groups)
        n_cols = min(max_cols, n_subjects)
        n_rows = ceil(n_subjects / n_cols)

        fig = make_subplots(
            rows=n_rows,
            cols=n_cols,
            specs=[[{"type": "scene"} for _ in range(n_cols)] for _ in range(n_rows)],
            subplot_titles=[f"Subject {subj}" for subj in subject_groups.keys()],
            horizontal_spacing=0.05,
            vertical_spacing=0.08,
        )

        # -------- 4. Add one surface per subject --------
        interp_method = "cubic"  # or "linear" / "nearest"

        for idx, (subj, df_sub) in enumerate(subject_groups.items()):
            row = idx // n_cols + 1
            col = idx % n_cols + 1

            x = df_sub[nutrient].values
            y = df_sub["epoch_num"].values
            z = df_sub["delta_auc"].values

            # Interpolate on the common grid
            try:
                zi = griddata((x, y), z, (xi_mesh, yi_mesh), method=interp_method)

                if np.isnan(zi).any():
                    zi_nearest = griddata((x, y), z, (xi_mesh, yi_mesh), method="nearest")
                    zi = np.where(np.isnan(zi), zi_nearest, zi)

                if np.all(np.isnan(zi)):
                    print(f"Subject {subj}: all NaNs after interpolation, skipping.")
                    continue

                zi = np.clip(zi, z_min, z_max)

            except Exception as e:
                print(f"Could not create surface for subject {subj}: {e}")
                continue

            # Show colorbar only on the first subject to avoid clutter
            show_colorbar = (idx == 0)

            fig.add_trace(
                go.Surface(
                    x=xi_mesh,
                    y=yi_mesh,
                    z=zi,
                    colorscale="Inferno",
                    opacity=0.9,
                    showscale=show_colorbar,
                    cmin=z_min,
                    cmax=z_max,
                    colorbar=dict(title="ΔAUC") if show_colorbar else None,
                    contours_z=dict(
                        show=True,
                        usecolormap=True,
                        project_z=True
                    ),
                ),
                row=row,
                col=col
            )

        # -------- 5. Shared axis ranges and layout --------
        scene_layout = dict(
            xaxis_title=f"{nutrient}",
            yaxis_title="Epoch Number",
            zaxis_title="ΔAUC",
            xaxis=dict(range=[x_min, x_max]),
            yaxis=dict(range=[y_min, y_max]),
            zaxis=dict(range=[z_min, z_max]),
            camera=dict(eye=dict(x=1.5, y=1.5, z=0.5)),
        )

        # Apply same layout to all scenes (scene, scene2, scene3, ...)
        scene_names = ["scene"] + [f"scene{i}" for i in range(2, n_subjects + 1)]
        fig.update_layout(
            **{name: scene_layout for name in scene_names},
            title=(
                f"{time_label}: {nutrient} vs Epoch vs ΔAUC "
                f"Per Subject ({physio_signal})"
            ),
            height=480 * n_rows,
            width=500 * n_cols,
            margin=dict(l=50, r=50, b=50, t=80),
        )

        out_path = os.path.join(
            self.config.out_dir_surface,
            f"{physio_signal}_{nutrient}_surface_per_subject_{time_label.replace(' ', '_')}.html"
        )
        fig.write_html(out_path)
        print(f"Saved per-subject 3D surface grid: {out_path}")



In [ ]:
#Run the Pipeline
config = PipelineConfig()
pipeline = PostprandialPipeline(config)
visualizer = PostprandialVisualizer(config)

for physio in config.physio_signals:
    print(f"\n=== Processing signal: {physio} ===")
    dataframes = pipeline.run_for_signal(physio)
    
    for nutrient in config.nutrients:
        if nutrient not in dataframes["All Data"].columns:
            print(f"Skipping nutrient {nutrient} (column not found)")
            continue

        #visualizer.plot_2d_by_time_of_day(dataframes, physio, nutrient)
        #visualizer.plot_3d_scatter(dataframes, physio, nutrient)
        #visualizer.plot_3d_surface(dataframes, physio, nutrient)
        visualizer.plot_3d_surface_by_subject(dataframes["All Data"], physio, nutrient, "All Data")
        visualizer.plot_3d_surface_by_subject(dataframes["Morning"], physio, nutrient, "Morning")
        visualizer.plot_3d_surface_by_subject(dataframes["Afternoon"], physio, nutrient, "Afternoon")
        visualizer.plot_3d_surface_by_subject(dataframes["Night"], physio, nutrient, "Night")

## Flexible Outlier Filtering (Per-Signal Thresholds and Z-score)

This cell defines a helper function `filter_outliers()` and a
`FilteredPostprandialPipeline` that:
- runs the original subject-level processing
- then removes outliers either using per-signal absolute thresholds
  and/or z-score filtering on `delta_value`.


In [16]:
def filter_outliers(
    df: pd.DataFrame,
    signal_col: str = "signal",
    delta_col: str = "delta_value",
    per_signal_thresholds: Optional[Dict[str, float]] = None,
    z_thresh: Optional[float] = None
) -> pd.DataFrame:
    """
    Filter outliers in delta_col per signal, using:
      - per-signal absolute thresholds (if provided), and/or
      - z-score thresholds within each signal group.

    If both per_signal_thresholds and z_thresh are provided, a row is kept only if:
        abs(delta) <= signal_threshold  AND  abs(zscore) <= z_thresh
    """
    if df.empty:
        return df

    def _filter_group(g: pd.DataFrame) -> pd.DataFrame:
        mask = pd.Series(True, index=g.index)

        # 1) per-signal absolute threshold
        if per_signal_thresholds is not None:
            sname = g[signal_col].iloc[0]
            thr = per_signal_thresholds.get(sname, None)
            if thr is not None:
                mask &= g[delta_col].abs() <= thr

        # 2) z-score threshold
        if z_thresh is not None and len(g) > 3:
            vals = g[delta_col]
            mu = vals.mean()
            sigma = vals.std(ddof=1)
            if sigma > 0:
                z = (vals - mu) / sigma
                mask &= z.abs() <= z_thresh

        return g[mask]

    return (
        df.groupby(signal_col, group_keys=False)
          .apply(_filter_group)
          .reset_index(drop=True)
    )


class FilteredPostprandialPipeline(PostprandialPipeline):
    """
    Extends the original PostprandialPipeline by applying outlier
    filtering to each of the returned DataFrames.
    """

    def __init__(
        self,
        config: PipelineConfig,
        per_signal_thresholds: Optional[Dict[str, float]] = None,
        z_thresh: Optional[float] = None
    ):
        super().__init__(config)
        self.per_signal_thresholds = per_signal_thresholds or {}
        self.z_thresh = z_thresh

    def run_for_signal(self, physio_signal: str) -> Dict[str, pd.DataFrame]:
        # First run the original pipeline
        dataframes = super().run_for_signal(physio_signal)

        # Then filter outliers on each DataFrame
        filtered = {}
        for label, df in dataframes.items():
            if df.empty:
                filtered[label] = df
            else:
                filtered[label] = filter_outliers(
                    df,
                    signal_col="signal",
                    delta_col="delta_value",
                    per_signal_thresholds=self.per_signal_thresholds,
                    z_thresh=self.z_thresh
                )
        return filtered


In [17]:
# Example per-signal thresholds for |delta_value|
signal_thresholds = {
    "glucose": 75.0,   # mg/dL·min
    "eda": 5.0,        # arbitrary AUC units
    "temp": 2.0,       # °C·min
    "bvp": 50.0,
    "acc_x": 5.0,
    "acc_y": 5.0,
    "acc_z": 5.0,
}

config = PipelineConfig() 
filtered_config = config  # or config for real data
visualizer = PostprandialVisualizer(config)

filtered_pipeline = FilteredPostprandialPipeline(
    filtered_config,
    per_signal_thresholds=signal_thresholds,
    z_thresh=3.0    # also enforce |z| <= 3 within each signal
)

"""
print("\n=== Running FILTERED pipeline on fake data (glucose only) ===")
filtered_dataframes = filtered_pipeline.run_for_signal("glucose")
display(filtered_dataframes["All Data"].head())

for physio in config.physio_signals:
    print(f"\n=== Processing signal: {physio} ===")

    for nutrient in config.nutrients:
        if nutrient not in filtered_dataframes["All Data"].columns:
            print(f"Skipping nutrient {nutrient} (column not found)")
            continue

        visualizer.plot_2d_by_time_of_day(filtered_dataframes, physio, nutrient)
        visualizer.plot_3d_scatter(filtered_dataframes, physio, nutrient)
        visualizer.plot_3d_surface(filtered_dataframes, physio, nutrient)
"""        

'\nprint("\n=== Running FILTERED pipeline on fake data (glucose only) ===")\nfiltered_dataframes = filtered_pipeline.run_for_signal("glucose")\ndisplay(filtered_dataframes["All Data"].head())\n\nfor physio in config.physio_signals:\n    print(f"\n=== Processing signal: {physio} ===")\n\n    for nutrient in config.nutrients:\n        if nutrient not in filtered_dataframes["All Data"].columns:\n            print(f"Skipping nutrient {nutrient} (column not found)")\n            continue\n\n        visualizer.plot_2d_by_time_of_day(filtered_dataframes, physio, nutrient)\n        visualizer.plot_3d_scatter(filtered_dataframes, physio, nutrient)\n        visualizer.plot_3d_surface(filtered_dataframes, physio, nutrient)\n'

In [ ]:
for physio in config.physio_signals:
    print(f"\n=== Processing signal: {physio} ===")
    dataframes = filtered_pipeline.run_for_signal(physio)
    
    for nutrient in config.nutrients:
        if nutrient not in dataframes["All Data"].columns:
            print(f"Skipping nutrient {nutrient} (column not found)")
            continue

        visualizer.plot_2d_by_time_of_day(dataframes, physio, nutrient)
        visualizer.plot_3d_scatter(dataframes, physio, nutrient)
        visualizer.plot_3d_surface(dataframes, physio, nutrient)

## Mini Fake Dataset for Sanity-Checking the Pipeline

This cell creates a small synthetic dataset that mimics the BIGIDEAs feature
pickles (one file per subject, with meal rows and per-signal *_segments DataFrames).
It then runs the pipeline on this toy data so we can verify that the end-to-end
processing and plotting work.


In [ ]:
import shutil
from datetime import datetime, timedelta

# Create a separate directory for fake test features
TEST_FEATURE_DIR = "./test_features"
os.makedirs(TEST_FEATURE_DIR, exist_ok=True)

# (Optional) clear out any old test files
for f in os.listdir(TEST_FEATURE_DIR):
    os.remove(os.path.join(TEST_FEATURE_DIR, f))

def make_fake_segments(start_time: pd.Timestamp,
                       duration_minutes: int = 120,
                       freq: str = "1T") -> pd.DataFrame:
    """
    Create a simple synthetic segment DataFrame with datetime index and
    columns for glucose / eda / temp / bvp / acc_x / acc_y / acc_z.
    """
    idx = pd.date_range(start=start_time, periods=duration_minutes, freq=freq)

    # Simple toy signals: slightly increasing glucose, noisy others
    glucose = 100 + np.linspace(0, 20, len(idx)) + np.random.normal(0, 2, len(idx))
    eda = 0.5 + 0.1 * np.sin(np.linspace(0, 4 * np.pi, len(idx))) + np.random.normal(0, 0.02, len(idx))
    temp = 33 + 0.2 * np.cos(np.linspace(0, 2 * np.pi, len(idx))) + np.random.normal(0, 0.05, len(idx))
    bvp = 1.0 + 0.05 * np.sin(np.linspace(0, 6 * np.pi, len(idx))) + np.random.normal(0, 0.01, len(idx))
    acc_x = np.random.normal(0, 0.1, len(idx))
    acc_y = np.random.normal(0, 0.1, len(idx))
    acc_z = np.random.normal(0, 0.1, len(idx))

    seg_df = pd.DataFrame({
        "glucose": glucose,
        "eda": eda,
        "temp": temp,
        "bvp": bvp,
        "acc_x": acc_x,
        "acc_y": acc_y,
        "acc_z": acc_z,
    }, index=idx)

    return seg_df

def make_fake_subject_pickle(subject_id: int, n_meals: int = 3):
    """
    Create a fake feature pickle for one subject.
    Each row represents one meal with:
    - timestamp
    - nutrient info (total_carb, sugar, protein, total_fat, calorie)
    - *_segments columns: each a small DataFrame with physio segments.
    """
    rows = []
    base_day = datetime(2025, 1, 1) + timedelta(days=subject_id)  # different day per subject

    for i in range(n_meals):
        # meal times at 08:00, 13:00, 19:00 or similar
        meal_time = base_day + timedelta(hours=8 + i * 5)

        seg_df = make_fake_segments(meal_time - timedelta(minutes=30), duration_minutes=150, freq="1T")

        # Random but reasonable macros
        total_carb = np.random.randint(20, 80)
        sugar = np.random.randint(5, 30)
        protein = np.random.randint(5, 40)
        total_fat = np.random.randint(5, 30)
        calorie = total_carb * 4 + protein * 4 + total_fat * 9

        row = {
            "timestamp": meal_time,
            "total_carb": total_carb,
            "sugar": sugar,
            "protein": protein,
            "total_fat": total_fat,
            "calorie": calorie,
            "glucose_segments": seg_df[["glucose"]].copy(),
            "eda_segments": seg_df[["eda"]].copy(),
            "temp_segments": seg_df[["temp"]].copy(),
            "bvp_segments": seg_df[["bvp"]].copy(),
            "acc_x_segments": seg_df[["acc_x"]].copy(),
            "acc_y_segments": seg_df[["acc_y"]].copy(),
            "acc_z_segments": seg_df[["acc_z"]].copy(),
        }
        rows.append(row)

    fake_df = pd.DataFrame(rows)
    out_path = os.path.join(TEST_FEATURE_DIR, f"segmented_df_{subject_id}_7200.pkl")
    fake_df.to_pickle(out_path)
    print(f"Saved fake subject file: {out_path}")


# ------------------------------------------------------------------
# 1) Generate fake pickles for a couple of test subjects
# ------------------------------------------------------------------
TEST_SUBJECT_IDS = (1, 2)

for sid in TEST_SUBJECT_IDS:
    make_fake_subject_pickle(sid, n_meals=3)

# ------------------------------------------------------------------
# 2) Create a test config that points to the fake directory
# ------------------------------------------------------------------
test_config = PipelineConfig(
    subject_ids=TEST_SUBJECT_IDS,
    feature_dir=TEST_FEATURE_DIR,
    file_pattern="segmented_df_{subject_id}_7200.pkl",  # matches names above
    post_meal_epochs=12,   # shorter for quick tests
    epoch_window_mins=5,
    outlier_threshold=75.0
)

test_pipeline = PostprandialPipeline(test_config)
test_visualizer = PostprandialVisualizer(test_config)

# ------------------------------------------------------------------
# 3) Run the pipeline on just glucose and one nutrient for a quick check
# ------------------------------------------------------------------
test_physio = "glucose"
test_nutrient = "sugar"

print(f"\n=== Running TEST pipeline for signal: {test_physio} ===")
test_dataframes = test_pipeline.run_for_signal(test_physio)

# Quick look at the combined DataFrame
display(test_dataframes["All Data"].head())

# Produce a small set of plots for sanity check
if test_nutrient in test_dataframes["All Data"].columns:
    test_visualizer.plot_2d_by_time_of_day(test_dataframes, test_physio, test_nutrient)
    test_visualizer.plot_3d_scatter(test_dataframes, test_physio, test_nutrient)
    test_visualizer.plot_3d_surface(test_dataframes, test_physio, test_nutrient)
else:
    print(f"Column {test_nutrient} not found in test data.")


In [ ]:
test_dataframes = test_pipeline.run_for_signal(test_physio)

print("All Data shape:", test_dataframes["All Data"].shape)
print("Columns:", list(test_dataframes["All Data"].columns))
print("Unique signals:", test_dataframes["All Data"].signal.unique() if not test_dataframes["All Data"].empty else [])
display(test_dataframes["All Data"].head())
